In [ ]:
!pip install torch torch-geometric scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.5 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import negative_sampling
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

# Load Cora dataset
dataset = Planetoid(root='data', name='Cora')
data = dataset[0]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- Model ----------
class UnifiedAttri2VecLSTM(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()

        # Attribute embedding
        self.attr_layer = nn.Linear(in_dim, hidden_dim)

        # Neighborhood embedding
        self.neigh_layer = nn.Linear(in_dim, hidden_dim)

        # Hierarchical fusion
        self.W_attr_star = nn.Linear(hidden_dim, hidden_dim)
        self.W_neigh_star = nn.Linear(hidden_dim, hidden_dim)

        # LSTM classifier
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, edge_pairs):

        # Attribute embedding
        E_attr = F.relu(self.attr_layer(x))

        # Neighborhood aggregation (mean aggregation)
        row, col = edge_index
        neigh_feat = torch.zeros_like(x)
        neigh_feat.index_add_(0, row, x[col])
        deg = torch.bincount(row, minlength=x.size(0)).float().unsqueeze(1)
        neigh_feat = neigh_feat / (deg + 1e-6)
        E_neigh = F.relu(self.neigh_layer(neigh_feat))

        # Unified embedding
        E_unified = torch.relu(
            self.W_attr_star(E_attr) + self.W_neigh_star(E_neigh)
        )

        # Pair embedding (Hadamard product)
        u, v = edge_pairs
        E_pair = E_unified[u] * E_unified[v]

        # LSTM expects sequence → treat each dimension as timestep
        seq = E_pair.unsqueeze(-1)
        _, (h, _) = self.lstm(seq)
        out = torch.sigmoid(self.fc(h[-1])).squeeze()

        return out

# ---------- Prepare data ----------
pos_edge_index = data.edge_index
neg_edge_index = negative_sampling(
    edge_index=data.edge_index,
    num_nodes=data.num_nodes,
    num_neg_samples=data.edge_index.size(1)
)

edge_pairs = torch.cat([pos_edge_index, neg_edge_index], dim=1)
labels = torch.cat([
    torch.ones(pos_edge_index.size(1)),
    torch.zeros(neg_edge_index.size(1))
]).to(device)

model = UnifiedAttri2VecLSTM(dataset.num_features, 64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

x = data.x.to(device)
edge_index = data.edge_index.to(device)
edge_pairs = edge_pairs.to(device)

# ---------- Training ----------
model.train()
for epoch in range(100):
    optimizer.zero_grad()
    pred = model(x, edge_index, edge_pairs)
    loss = F.binary_cross_entropy(pred, labels)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# ---------- Evaluation ----------
model.eval()
with torch.no_grad():
    pred = model(x, edge_index, edge_pairs).cpu().numpy()
    y_true = labels.cpu().numpy()
    y_pred = (pred > 0.5).astype(int)

auc = roc_auc_score(y_true, pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("\n===== Results on Cora =====")
print("AUC:", auc)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)


Processing...
Done!


Epoch 0, Loss: 0.6932
Epoch 10, Loss: 0.5932
Epoch 20, Loss: 0.4947
Epoch 30, Loss: 0.4163
Epoch 40, Loss: 0.3507
Epoch 50, Loss: 0.2856
Epoch 60, Loss: 0.2301
Epoch 70, Loss: 0.1739
Epoch 80, Loss: 0.1478
Epoch 90, Loss: 0.2311

===== Results on Cora =====
AUC: 0.9799913552232873
Precision: 0.9498474446987033
Recall: 0.9437286851079955
F1-score: 0.9467781790534119
